# K-Nearest Neighbor (KNN) - Day 2 (Scikit-Learn)

## Statistical Methods for Selecting k

### 1. Cross-Validation

Cross-Validation is a good way to find the best value of k. This means dividing the dataset into k parts. The model is trained on some of these parts and tested on the remaining ones. This process is repeated for each part.

### 2. Elbow Method

In Elbow Method we draw a graph showing the error rate or accuracy for different k values. As k increases the error usually drops at first. But after a certain point error stops decreasing quickly. The point where the curve changes direction and looks like an "elbow" is usually the best choice for k.

### 3. Odd Values for k

It's a good idea to use an odd number for k, especially in classification problems. This helps avoid ties when deciding which class is the most common among the neighbors.

---

## Distance Metrics Used in KNN Algorithm

### 1. Euclidean Distance

$$d(x, X_i) = \sqrt{\sum_{j=1}^{n} (x_j - X_{ij})^2}$$

### 2. Manhattan Distance

$$d(x, y) = \sum_{i=1}^{n} |x_i - y_i|$$

### 3. Minkowski Distance

$$d(x, y) = \left( \sum_{i=1}^{n} |x_i - y_i|^p \right)^{\frac{1}{p}}$$

- When p=2 → Euclidean
- When p=1 → Manhattan

---

## One-Line Summary

**Select optimal k using cross-validation and elbow method, with distance metrics like Euclidean, Manhattan, or Minkowski.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

print("="*50)
print("KNN DAY 2 - SELECTING K (SCIKIT-LEARN)")
print("="*50)

In [ ]:
# Create dataset
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, random_state=42)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

## Elbow Method - Cross-Validation for Different k

In [ ]:
# Cross-validation for different k values
k_values = list(range(1, 31))
cv_scores = []
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    
    # Cross-validation (5-fold)
    cv_score = cross_val_score(knn, X_train, y_train, cv=5).mean()
    cv_scores.append(cv_score)
    
    # Train on full training set
    knn.fit(X_train, y_train)
    train_scores.append(knn.score(X_train, y_train))
    test_scores.append(knn.score(X_test, y_test))

print("Cross-validation completed for k = 1 to 30")

In [ ]:
# Elbow plot
plt.figure(figsize=(12, 6))
plt.plot(k_values, cv_scores, marker='o', label='Cross-Validation Accuracy', color='green')
plt.plot(k_values, train_scores, marker='s', label='Train Accuracy', color='blue', alpha=0.5)
plt.plot(k_values, test_scores, marker='d', label='Test Accuracy', color='red', alpha=0.5)
plt.xlabel('K Value')
plt.ylabel('Accuracy')
plt.title('Elbow Method - Finding Optimal K (with Cross-Validation)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Find best k
best_k = k_values[np.argmax(cv_scores)]
best_cv_score = max(cv_scores)

print(f"Best k (by cross-validation): {best_k}")
print(f"Cross-validation accuracy: {best_cv_score:.4f}")

# Train final model with best k
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train, y_train)
final_test_acc = best_knn.score(X_test, y_test)
print(f"Test accuracy with k={best_k}: {final_test_acc:.4f}")

## Distance Metrics Comparison

In [ ]:
print("\n" + "="*50)
print("Distance Metrics Comparison")
print("="*50)

# Try different metrics
metrics = ['euclidean', 'manhattan', 'minkowski']
p_values = [2, 1, 3]  # p=2 for euclidean, p=1 for manhattan, p=3 for custom

for i, metric in enumerate(metrics):
    knn = KNeighborsClassifier(n_neighbors=best_k, metric=metric, p=p_values[i])
    knn.fit(X_train, y_train)
    acc = knn.score(X_test, y_test)
    print(f"Metric = {metric}: Accuracy = {acc:.4f}")

## Weights Comparison

In [ ]:
print("\n" + "="*50)
print("Weights Comparison")
print("="*50)

weights = ['uniform', 'distance']

for weight in weights:
    knn = KNeighborsClassifier(n_neighbors=best_k, weights=weight)
    knn.fit(X_train, y_train)
    acc = knn.score(X_test, y_test)
    print(f"Weights = {weight}: Accuracy = {acc:.4f}")

In [ ]:
# Visualize decision boundary with best k
def plot_knn_boundary(knn, X, y, title):
    plt.figure(figsize=(8, 6))
    
    # Plot points
    plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='bwr', edgecolors='black')
    
    # Decision boundary
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.2, cmap='bwr')
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.show()

plot_knn_boundary(best_knn, X_test, y_test, f"KNN Decision Boundary (k={best_k})")

In [ ]:
# Day 2 Completed
print("\n" + "="*50)
print("KNN DAY 2 COMPLETED")
print("="*50)
print("Topics covered:")
print("- Cross-Validation for selecting k")
print("- Elbow Method with cv_scores")
print("- Distance metrics (Euclidean, Manhattan, Minkowski)")
print("- Weights comparison (uniform vs distance)")
print("- Finding optimal k with sklearn")
print("="*50)"
print("\nNext: Day 3 - KNN on Real Dataset (Drug Classifier)")